In [ ]:
# ==============================================================================
# 2_feature_engineering.ipynb
# ==============================================================================
from pyspark.ml.feature import VectorAssembler, StandardScaler

print("\n>>> STEP 3: Strategic Sampling & Memory Caching...")

# 1. Aggressive Upstream Sampling
vraj_sample_df = df_raw.sample(fraction=0.05, seed=42)

# 2. Custom Transformer Pipeline
feature_cols = [f"feature_{i}" for i in range(1, 29)]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")
scaler = StandardScaler(inputCol="raw_features", outputCol="features", withStd=True, withMean=True)

df_assembled = assembler.transform(vraj_sample_df)
df_scaled = scaler.fit(df_assembled).transform(df_assembled)

# 3. Memory Management
model_data = df_scaled.select("features", "label").cache()
print(f"   [CACHED] Data ready for ML Training: {model_data.count()} rows")

# Split the data
train_data, test_data = model_data.randomSplit([0.8, 0.2], seed=42)
print("   Data successfully split into Train (80%) and Test (20%).")